# On-Device 문제-답-해설 통합본

문제 노트북과 답지 노트북을 비교해, 실습 문제와 직접 관련 있는 셀만 추렸습니다. 각 항목은 `문제 -> 채워야 할 답 -> 간략 해설` 순서입니다.


## 1. Pruning for CNN.ipynb

### 문제 1. [실습 1] Fine-grained Pruning 구현

- 원본 문제 파일/셀: `1. Pruning for CNN.ipynb` / code cell `28`
- 원본 답지 파일/셀: `1. Pruning for CNN_answer.ipynb` / code cell `29`


## [실습 1] Fine-grained Pruning 구현

Fine-grained Pruning은 신경망의 가중치를 개별적으로 제거하는 기법입니다. 각 가중치의 크기(Magnitude)를 기준으로 중요도를 평가하고, 지정된 희소도(sparsity)에 따라 중요도가 낮은 가중치들을 0으로 만듭니다. 이를 통해 모델의 크기를 줄이고 연산량을 감소시킬 수 있습니다.

아래 빈칸을 채워 `prune_weight_fine_grained` 함수를 완성하세요.

In [ ]:
def prune_weight_fine_grained(weight: torch.Tensor, sparsity: float) -> None:
    """가중치 텐서에 대해 fine-grained pruning을 수행하는 함수

    Args:
        weight: pruning할 가중치 텐서
        sparsity: pruning할 비율 (0~1 사이 값)

    Returns:
        pruning mask 텐서
    """
    # sparsity 값을 0~1 사이로 제한
    sparsity = min(1.0, max(0.0, sparsity))

    # 특수한 경우 처리
    if sparsity == 1.0:  # 모든 가중치를 제거
        weight.zero_()
        return torch.zeros_like(weight)
    elif sparsity == 0.0:  # 모든 가중치를 유지
        return torch.ones_like(weight)

    ##################### YOUR CODE STARTS HERE #####################
    # 제거할 원소 개수를 계산하세요.
    # hint: round() 함수를 사용하세요.
    num_pruned_elements =

    # 가중치의 중요도를 절댓값으로 importance 계산
    # hint: torch.abs() 함수를 사용하세요.
    importance =

    # pruning trheshold를 계산하세요.
    # hint: torch.kthvalue() 함수를 사용하세요.
    threshold =

    # threshold보다 큰 값들은 유지(1), 작은 값들은 제거(0)하는 마스크 생성
    # hint: 부등호를 사용하세요.
    mask =
    ##################### YOUR CODE ENDS HERE #######################

    # 마스크를 적용하여 pruning 수행
    weight.mul_(mask)

    return mask

# 마스크 생성 및 시각화
mask_fine_grained = prune_weight_fine_grained(weight.clone(), prune_sparsity)
draw_weight_distribution(mask_fine_grained, title="Fine-grained Pruning Mask")

#### 채워야 할 답

1. `num_pruned_elements = round(weight.numel() * sparsity)`
2. `importance = torch.abs(weight)`
3. `threshold = torch.kthvalue(importance.flatten(), num_pruned_elements)[0]`
4. `mask = importance > threshold`

#### 간략 해설

1. 목표 sparsity에 맞춰 제거할 weight 개수를 계산합니다.
2. Magnitude pruning은 절댓값이 작은 weight를 덜 중요하다고 보므로 절댓값을 importance로 사용합니다.
3. 목표 sparsity에 맞춰 제거할 weight 개수를 계산합니다.
4. threshold보다 큰 항목만 남기고 나머지는 pruning하기 위한 boolean mask입니다.


### 문제 2. [실습 2] Sensitivity Analysis를 통한 Pruning 수행

- 원본 문제 파일/셀: `1. Pruning for CNN.ipynb` / code cell `47`
- 원본 답지 파일/셀: `1. Pruning for CNN_answer.ipynb` / code cell `48`


## [실습 2] Sensitivity Analysis를 통한 Pruning 수행

Sensitivity analysis 결과를 바탕으로 각 레이어별 sparsity를 `custom_sparsity_dict`에 입력하세요.

위 그래프의 분석 결과를 토대로 아래와 같이 진행하면 됩니다다:
- 정확도 하락이 적은 레이어: 높은 sparsity 값 설정
- 정확도 하락이 큰 레이어: 낮은 sparsity 값 설정

이러한 레이어별 차등적인 pruning을 통해 모델의 전반적인 성능은 유지하면서 효율적인 압축을 달성하는 것이 목표입니다.

In [ ]:
def get_model_sparsity(model: nn.Module) -> float:
    """
    모델 전체의 sparsity 계산.

    Args:
        model: 스파시티를 계산할 모델

    Returns:
        float: 모델의 전체 스파시티 비율 (0~1 사이 값)
    """
    num_nonzeros, num_elements = 0, 0
    for param in model.parameters():
        num_nonzeros += param.count_nonzero()
        num_elements += param.numel()
    return 1 - float(num_nonzeros) / num_elements

custom_sparsity_dict = {
##################### YOUR CODE STARTS HERE #####################
    # Key는 수정하지 마시고, 각 layer의 sparsity 값을 조정하세요.
    'backbone.conv0.weight': ,
    'backbone.conv1.weight': ,
    'backbone.conv2.weight': ,
    'backbone.conv3.weight': ,
    'backbone.conv4.weight': ,
    'backbone.conv5.weight': ,
    'backbone.conv6.weight': ,
    'backbone.conv7.weight': ,
    'classifier.weight':
##################### YOUR CODE ENDS HERE #######################
}

# 프루닝 적용 및 평가
pruner = FineGrainedPruner(model, custom_sparsity_dict)
pruner.apply(model)
custom_pruned_model_accuracy = evaluate(model, dataloader['test'])
custom_sparsity = get_model_sparsity(model)
model.recover_model()

# 결과 출력
print(f"custom_pruned_model has accuracy={custom_pruned_model_accuracy:.2f}%")
print(f"Model sparsity: {custom_sparsity:.3f}")

#### 채워야 할 답

1. `'backbone.conv0.weight': 0.5,`
2. `'backbone.conv1.weight': 0.8,`
3. `'backbone.conv2.weight': 0.8,`
4. `'backbone.conv3.weight': 0.7,`
5. `'backbone.conv4.weight': 0.7,`
6. `'backbone.conv5.weight': 0.8,`
7. `'backbone.conv6.weight': 0.8,`
8. `'backbone.conv7.weight': 0.9,`
9. `'classifier.weight': 0.9`

#### 간략 해설

1. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.
2. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.
3. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.
4. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.
5. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.
6. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.
7. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.
8. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.
9. 레이어별 민감도에 따라 서로 다른 sparsity를 지정하는 설정값입니다.


### 문제 3. [실습 3] Global Magnitude Pruning 구현

- 원본 문제 파일/셀: `1. Pruning for CNN.ipynb` / code cell `53`
- 원본 답지 파일/셀: `1. Pruning for CNN_answer.ipynb` / code cell `54`


## [실습 3] Global Magnitude Pruning 구현

이 실습에서는 Global Magnitude Pruning을 구현합니다. Global Magnitude Pruning은 모델의 모든 가중치를 하나의 집합으로 간주하고, 가중치의 절대값을 기준으로 중요도를 평가하여 전체 모델에서 가장 작은 가중치를 제거하는 방식입니다. 이는 각 레이어별로 균일한 비율로 프루닝하는 것이 아니라, 전체 모델에서 가중치의 크기에 따라 프루닝을 수행합니다.

**구현 과정:**
1. 모든 레이어의 가중치를 하나의 벡터로 통합
2. 가중치의 절대값으로 중요도 계산
3. 프루닝 비율에 따른 임계값 설정
4. 임계값 미만의 가중치를 제거하는 마스크 생성 및 적용

이러한 방식으로 모델의 파라미터를 효율적으로 줄이면서도 중요한 정보는 보존할 수 있습니다. 결과적으로 모델 크기와 연산량을 감소시키면서 성능 저하를 최소화하는 것이 목표입니다.

아래 빈칸을 채워 `FineGrainedPrunerV2` 클래스를 완성하세요.

In [ ]:
class FineGrainedPrunerV2:
    def __init__(self, model, sparsity, global_prune=False):
        """
        전역 또는 레이어별 프루닝을 위한 프루너 클래스

        Args:
            model: 프루닝할 모델
            sparsity: 프루닝 비율 (0~1)
            global_prune: 전역 프루닝 여부
        """
        self.masks = FineGrainedPrunerV2.prune(model, sparsity, global_prune)

    @torch.no_grad()
    def apply(self, model):
        """프루닝 마스크를 모델에 적용"""
        for name, param in model.named_parameters():
            if name in self.masks:
                param *= self.masks[name]

    @staticmethod
    @torch.no_grad()
    def prune(model, sparsity, global_prune):
        """
        전역 또는 레이어별 프루닝 수행

        Args:
            model: 프루닝할 모델
            sparsity: 프루닝 비율 (0~1)
            global_prune: 전역 프루닝 여부

        Returns:
            masks: 프루닝 마스크 딕셔너리
        """
        masks = dict()
        if global_prune:
            # 모든 2D 이상의 파라미터를 1차원으로 변환하여 수집
            parameters_to_prune = []
            for name, param in model.named_parameters():
                if param.dim() > 1:  # conv, fc 레이어만 프루닝
                    parameters_to_prune.append(param.view(-1))

            ##################### YOUR CODE STARTS HERE #####################
            # 모든 weight를 하나의 텐서로 결합해주세요..
            # hint: torch.cat()을 사용하세요.
            all_weights =

            # all_weights를 대상으로 global threshold를 구해주세요.
            num_elements =
            num_zeros =
            importance =
            threshold =
            ##################### YOUR CODE ENDS HERE #######################

            # threshold 기반 마스크 생성
            for name, param in model.named_parameters():
                if param.dim() > 1:
                    mask = torch.abs(param.data) > threshold
                    masks[name] = mask
        else:
            # 레이어별 프루닝 수행
            for name, param in model.named_parameters():
                if param.dim() > 1: # we only prune conv and fc weights
                    masks[name] = prune_weight_fine_grained(param, sparsity)
        return masks

#### 채워야 할 답

1. `all_weights = torch.cat(parameters_to_prune)`
2. `num_elements = all_weights.numel()`
3. `num_zeros = round(num_elements * sparsity)`
4. `importance = torch.abs(all_weights)`
5. `threshold = torch.kthvalue(importance, num_zeros)[0]`

#### 간략 해설

1. Global pruning은 layer별이 아니라 전체 weight를 한 번에 비교하므로 모든 대상 weight를 연결합니다.
2. 문제 셀에서 비워진 핵심 구현 줄이며, 답지에서는 이 줄로 해당 알고리즘 단계를 완성합니다.
3. 목표 sparsity에 맞춰 제거할 weight 개수를 계산합니다.
4. Magnitude pruning은 절댓값이 작은 weight를 덜 중요하다고 보므로 절댓값을 importance로 사용합니다.
5. 목표 sparsity에 맞춰 제거할 weight 개수를 계산합니다.


### 문제 4. [실습 4] Sparsity Scheduler 구현

- 원본 문제 파일/셀: `1. Pruning for CNN.ipynb` / code cell `64`
- 원본 답지 파일/셀: `1. Pruning for CNN_answer.ipynb` / code cell `65`


## [실습 4] Sparsity Scheduler 구현

이 실습에서는 시간에 따른 Pruning 비율을 조절하는 다음 2가지 Sparsity Scheduler를 구현합니다:
- **Linear sparsity scheduler**: $v^{(t)} = v_f + \left(v_i - v_f\right)\left(1 - \frac{t - t_i}{t_f - t_i}\right)$
    - 시간에 따라 선형적으로 sparsity가 증가
- **Cubic sparsity scheduler**: $v^{(t)} = v_f + \left(v_i - v_f\right)\left(1 - \frac{t - t_i}{t_f - t_i}\right)^3$
    - 시간에 따라 세제곱 비율로 sparsity가 증가

아래 빈 칸을 채워 `get_sparsity_schedule` 함수를 완성하세요.

In [ ]:
def get_sparsity_schedule(num_epochs: int,
                          epoch_start: int,
                          epoch_end: int,
                          sparsity_start: float,
                          sparsity_end: float,
                          exponent: int,
                          ) -> List[float]:
    """
    스파시티 스케줄을 생성하는 함수입니다.

    Args:
        num_epochs: 전체 에폭 수
        epoch_start: 스파시티 스케줄링 시작 에폭
        epoch_end: 스파시티 스케줄링 종료 에폭
        sparsity_start: 시작 스파시티 값
        sparsity_end: 종료 스파시티 값
        exponent: 스케줄링 지수 (1: linear, 3: cubic)

    Returns:
        각 에폭별 스파시티 값 리스트
    """
    sparsity_schedule = []
    ##################### YOUR CODE STARTS HERE #####################
    for epoch in range(num_epochs):
        # 위 수식에 맞게 sparsity를 계산하세요. (단, epoch_start보다 작은 경우 0으로 epochs_end 이후는 sparsity_end로 설정합니다.)
        if epoch < epoch_start:
            sparsity =
        elif epoch >= epoch_end:
            sparsity =
        else:
            sparsity =

        sparsity_schedule.append(sparsity)
    ##################### YOUR CODE ENDS HERE #######################
    return sparsity_schedule

# Linear sparsity schedule
# linear_sparsity_schedule = [0.9, 0.925, 0.95, 0.95, 0.95]
linear_sparsity_schedule = get_sparsity_schedule(num_finetune_epochs, 0, 2, 0.9, target_sparsity, 1)
print("Linear sparsity schedule:", linear_sparsity_schedule)

# Cubic sparsity schedule
# cubic_sparsity_schedule = [0.9, 0.94375, 0.95, 0.95, 0.95]
cubic_sparsity_schedule = get_sparsity_schedule(num_finetune_epochs, 0, 2, 0.9, target_sparsity, 3)
print("Cubic sparsity schedule:", cubic_sparsity_schedule)

#### 채워야 할 답

1. `sparsity = 0`
2. `sparsity = sparsity_end`
3. `sparsity = sparsity_end + (sparsity_start - sparsity_end) * (1 - (epoch - epoch_start) / (epoch_end - epoch_start)) ** exponent`

#### 간략 해설

1. epoch 위치에 따라 pruning 강도를 조절하는 scheduler 식입니다.
2. epoch 위치에 따라 pruning 강도를 조절하는 scheduler 식입니다.
3. epoch 위치에 따라 pruning 강도를 조절하는 scheduler 식입니다.


## 2. Quantization for CNN.ipynb

### 문제 5. [실습 1] Linear quantization 구현

- 원본 문제 파일/셀: `2. Quantization for CNN.ipynb` / code cell `24`
- 원본 답지 파일/셀: `2. Quantization for CNN_answer.ipynb` / code cell `24`


## [실습 1] Linear quantization 구현

다음 linear quantization 함수를 완성해 주세요.

**Hint**:
*   $r=S(q-Z)$ 로 부터, $q = r/S + Z$ 를 도출할 수 있습니다.
*   $r$ 과 $S$ 는 모두 부동 소수점이므로, 정수 $Z$ 을 부동 소수점 $r/S$ 에 직접 더할 수 없습니다. 그러므로 $q = \mathrm{int}(\mathrm{round}(r/S)) + Z$ 와 같이 계산해야 합니다.
*   [`torch.FloatTensor`](https://pytorch.org/docs/stable/tensors.html) 에서 [`torch.IntTensor`](https://pytorch.org/docs/stable/tensors.html) 로 변환하려면, 먼저 [`torch.round()`](https://pytorch.org/docs/stable/generated/torch.round.html#torch.round), [`torch.Tensor.round()`](https://pytorch.org/docs/stable/generated/torch.Tensor.round.html#torch.Tensor.round), 또는 [`torch.Tensor.round_()`](https://pytorch.org/docs/stable/generated/torch.Tensor.round_)을 사용하여 모든 값을 부동 소수점 정수로 반올림 한 후, [`torch.Tensor.to(torch.int8)`](https://pytorch.org/docs/stable/generated/torch.Tensor.to.html#torch.Tensor.to) 를 사용하여 데이터 유형을 [`torch.float`](https://pytorch.org/docs/stable/tensors.html) 에서 [`torch.int8`](https://pytorch.org/docs/stable/tensors.html)로 변환해야 합니다.

In [ ]:
def linear_quantize(fp_tensor, bitwidth, scale, zero_point, dtype=torch.int8) -> torch.Tensor:
    """
    linear quantization for single fp_tensor
      from
        fp_tensor = (quantized_tensor - zero_point) * scale
      we have,
        quantized_tensor = int(round(fp_tensor / scale)) + zero_point
    :param tensor: [torch.(cuda.)FloatTensor] floating tensor to be quantized
    :param bitwidth: [int] quantization bit width
    :param scale: [torch.(cuda.)FloatTensor] scaling factor
    :param zero_point: [torch.(cuda.)IntTensor] the desired centroid of tensor values
    :return:
        [torch.(cuda.)FloatTensor] quantized tensor whose values are integers
    """
    assert(fp_tensor.dtype == torch.float)
    assert(isinstance(scale, float) or
           (scale.dtype == torch.float and scale.dim() == fp_tensor.dim()))
    assert(isinstance(zero_point, int) or
           (zero_point.dtype == dtype and zero_point.dim() == fp_tensor.dim()))

    ############### YOUR CODE STARTS HERE ###############
    # Step 1: fp_tensor를 scale 하세요.
    scaled_tensor =
    # Step 2: 부동 소수점 값을 정수 값으로 rounding 하세요.
    rounded_tensor =
    ############### YOUR CODE ENDS HERE #################

    rounded_tensor = rounded_tensor.to(dtype)

    ############### YOUR CODE STARTS HERE ###############
    # Step 3: rounded_tensor를 zero_point 만큼 shift하여 영점을 조정합니다.
    shifted_tensor =
    ############### YOUR CODE ENDS HERE #################

    # Step 4: shifted_tensor를 bitwidth 범위에 있도록 절사(clamp)합니다.
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    quantized_tensor = shifted_tensor.clamp_(quantized_min, quantized_max)
    return quantized_tensor

#### 채워야 할 답

1. `scaled_tensor = fp_tensor/scale`
2. `rounded_tensor = torch.round(scaled_tensor)`
3. `shifted_tensor = rounded_tensor + zero_point`

#### 간략 해설

1. 실수 tensor를 quantization scale로 나누어 integer grid 기준 값으로 변환합니다.
2. 실수 tensor를 quantization scale로 나누어 integer grid 기준 값으로 변환합니다.
3. 양자화 값은 정수 level에 매핑되어야 하므로 반올림합니다.


### 문제 6. [실습 2] Scale 및 zero point 계산

- 원본 문제 파일/셀: `2. Quantization for CNN.ipynb` / code cell `33`
- 원본 답지 파일/셀: `2. Quantization for CNN_answer.ipynb` / code cell `33`


# [실습 2] Scale 및 zero point 계산

부동 소수점 텐서 $r$ 에서 scale $S$ 와 zero point $Z$ 을 계산하는 다음 함수를 완성해 주세요.

In [ ]:
def get_quantization_scale_and_zero_point(fp_tensor, bitwidth):
    """
    get quantization scale for single tensor
    :param fp_tensor: [torch.(cuda.)Tensor] floating tensor to be quantized
    :param bitwidth: [int] quantization bit width
    :return:
        [float] scale
        [int] zero_point
    """
    quantized_min, quantized_max = get_quantized_range(bitwidth)
    fp_max = fp_tensor.max().item()
    fp_min = fp_tensor.min().item()

    ############### YOUR CODE STARTS HERE ###############
    # hint: quantized_max - quantized_min = 2 ** bitwith - 1
    scale =
    zero_point =
    ############### YOUR CODE ENDS HERE #################

    # clip the zero_point to fall in [quantized_min, quantized_max]
    if zero_point < quantized_min:
        zero_point = quantized_min
    elif zero_point > quantized_max:
        zero_point = quantized_max
    else: # convert from float to int using round()
        zero_point = round(zero_point)
    return scale, int(zero_point)

#### 채워야 할 답

1. `scale = (fp_max - fp_min) / (quantized_max - quantized_min)`
2. `zero_point = round(quantized_min - fp_min/scale)`

#### 간략 해설

1. 실수 범위와 정수 표현 범위의 비율이 quantization scale입니다.
2. zero point를 더해 비대칭 quantization의 정수 범위로 이동시킵니다.


### 문제 7. [실습 3] Quantized FC layer 구현

- 원본 문제 파일/셀: `2. Quantization for CNN.ipynb` / code cell `55`
- 원본 답지 파일/셀: `2. Quantization for CNN_answer.ipynb` / code cell `55`


# [실습 3] Quantized FC layer 구현

Quantized fully-connected layer를 추론하기 위해 다음 함수를 완성해 주세요.

**Hint**:

> $q_{\mathrm{output}} = (\mathrm{Linear}[q_{\mathrm{input}}, q_{\mathrm{weight}}] + Q_{\mathrm{bias}})\cdot (S_{\mathrm{input}} S_{\mathrm{weight}} / S_{\mathrm{output}}) + Z_{\mathrm{output}}$

In [ ]:
def quantized_linear(input, weight, bias, feature_bitwidth, weight_bitwidth,
                     input_zero_point, output_zero_point,
                     input_scale, weight_scale, output_scale):
    """
    quantized fully-connected layer
    :param input: [torch.CharTensor] quantized input (torch.int8)
    :param weight: [torch.CharTensor] quantized weight (torch.int8)
    :param bias: [torch.IntTensor] shifted quantized bias or None (torch.int32)
    :param feature_bitwidth: [int] quantization bit width of input and output
    :param weight_bitwidth: [int] quantization bit width of weight
    :param input_zero_point: [int] input zero point
    :param output_zero_point: [int] output zero point
    :param input_scale: [float] input feature scale
    :param weight_scale: [torch.FloatTensor] weight per-channel scale
    :param output_scale: [float] output feature scale
    :return:
        [torch.CharIntTensor] quantized output feature (torch.int8)
    """
    assert(input.dtype == torch.int8)
    assert(weight.dtype == input.dtype)
    assert(bias is None or bias.dtype == torch.int32)
    assert(isinstance(input_zero_point, int))
    assert(isinstance(output_zero_point, int))
    assert(isinstance(input_scale, float))
    assert(isinstance(output_scale, float))
    assert(weight_scale.dtype == torch.float)

    # Step 1: integer-based fully-connected (8-bit multiplication with 32-bit accumulation)
    if 'cpu' in input.device.type:
        # use 32-b MAC for simplicity
        output = torch.nn.functional.linear(input.to(torch.int32), weight.to(torch.int32), bias)
    else:
        # 현재 버전의 PyTorch는 GPU에서 정수 기반의 linear() 연산을 아직 지원하지 않습니다.
        output = torch.nn.functional.linear(input.float(), weight.float(), bias.float())

    # weight scale의 shape 가[oc, 1, 1, 1] 인 반면, output의 shape는 [batch_size, oc]이므로 shape를 조정합니다.
    weight_scale = weight_scale.flatten().view(1, -1)

    # scale이 부동 소수점이므로, 출력도 부동 소수점으로 변환해야 합니다.
    output = output.float()

    ############### YOUR CODE STARTS HERE ###############
    # Step 2: output 텐서를 scale합니다.
    output =

    # Step 3: output을 out_zero_point 만큼 이동하여 영점을 조정합니다.
    output =
    ############### YOUR CODE ENDS HERE #################

    # 모든 값이 지정된 비트폭 범위 내에 있도록 clamp합니다.
    output = output.round().clamp(*get_quantized_range(feature_bitwidth)).to(torch.int8)
    return output

#### 채워야 할 답

1. `output = output*(input_scale * weight_scale / output_scale)`
2. `output = output + output_zero_point`

#### 간략 해설

1. integer 연산 결과를 입력/weight/output scale 비율로 재스케일링합니다.
2. 출력 tensor를 output quantization 범위의 zero point 기준으로 이동합니다.


### 문제 8. [실습 3] Quantized convolution layer 구현

- 원본 문제 파일/셀: `2. Quantization for CNN.ipynb` / code cell `63`
- 원본 답지 파일/셀: `2. Quantization for CNN_answer.ipynb` / code cell `63`


# [실습 3] Quantized convolution layer 구현

Quantized convolution layer를 추론하기 위해 다음 함수를 완성해 주세요.

**Hint**:
> $q_{\mathrm{output}} = (\mathrm{CONV}[q_{\mathrm{input}}, q_{\mathrm{weight}}] + Q_{\mathrm{bias}}) \cdot (S_{\mathrm{input}}S_{\mathrm{weight}} / S_{\mathrm{output}}) + Z_{\mathrm{output}}$

In [ ]:
def quantized_conv2d(input, weight, bias, feature_bitwidth, weight_bitwidth,
                     input_zero_point, output_zero_point,
                     input_scale, weight_scale, output_scale,
                     stride, padding, dilation, groups):
    """
    quantized 2d convolution
    :param input: [torch.CharTensor] quantized input (torch.int8)
    :param weight: [torch.CharTensor] quantized weight (torch.int8)
    :param bias: [torch.IntTensor] shifted quantized bias or None (torch.int32)
    :param feature_bitwidth: [int] quantization bit width of input and output
    :param weight_bitwidth: [int] quantization bit width of weight
    :param input_zero_point: [int] input zero point
    :param samp: [int] output zero point
    :param input_scale: [float] input feature scale
    :param weight_scale: [torch.FloatTensor] weight per-channel scale
    :param output_scale: [float] output feature scale
    :return:
        [torch.(cuda.)CharTensor] quantized output feature
    """
    assert(len(padding) == 4)
    assert(input.dtype == torch.int8)
    assert(weight.dtype == input.dtype)
    assert(bias is None or bias.dtype == torch.int32)
    assert(isinstance(input_zero_point, int))
    assert(isinstance(output_zero_point, int))
    assert(isinstance(input_scale, float))
    assert(isinstance(output_scale, float))
    assert(weight_scale.dtype == torch.float)

    # Step 1: calculate integer-based 2d convolution (8-bit multiplication with 32-bit accumulation)
    input = torch.nn.functional.pad(input, padding, 'constant', input_zero_point)
    if 'cpu' in input.device.type:
        # use 32-b MAC for simplicity
        output = torch.nn.functional.conv2d(input.to(torch.int32), weight.to(torch.int32), None, stride, 0, dilation, groups)
    else:
        # 현재 버전의 PyTorch는 GPU에서 정수 기반의 conv2d() 연산을 아직 지원하지 않습니다.
        output = torch.nn.functional.conv2d(input.float(), weight.float(), None, stride, 0, dilation, groups)
        output = output.round().to(torch.int32)
    if bias is not None:
        output = output + bias.view(1, -1, 1, 1)


    # weight scale의 shape가 [oc, 1, 1, 1]인 반면, output의 shape는 [batch_size, oc, height, width]이므로 shape를 조정합니다.
    weight_scale = weight_scale.flatten().view(1, -1, 1, 1)

    # scale이 부동 소수점이므로, 출력도 부동 소수점으로 변환해야 합니다.
    output = output.float()

    ############### YOUR CODE STARTS HERE ###############
    # hint: 이번 코드 블록은 quantized_linear()와 매우 유사합니다.

    # Step 2: output 텐서를 scale합니다.
    output =

    # Step 3: output을 out_zero_point 만큼 이동하여 영점을 조정합니다.
    output =
    ############### YOUR CODE ENDS HERE #################

    # 모든 값이 지정된 비트폭 범위 내에 있도록 clamp 합니다.
    output = output.round().clamp(*get_quantized_range(feature_bitwidth)).to(torch.int8)
    return output

#### 채워야 할 답

1. `output = output*(input_scale * weight_scale / output_scale)`
2. `output = output + output_zero_point`

#### 간략 해설

1. integer 연산 결과를 입력/weight/output scale 비율로 재스케일링합니다.
2. 출력 tensor를 output quantization 범위의 zero point 기준으로 이동합니다.


### 문제 9. [실습 4] K-means quantization 구현

- 원본 문제 파일/셀: `2. Quantization for CNN.ipynb` / code cell `85`
- 원본 답지 파일/셀: `2. Quantization for CNN_answer.ipynb` / code cell `85`


# [실습 4] K-means quantization 구현

다음 K-means quantization 함수를 완성해 주세요.

In [ ]:
def k_means_quantize(fp32_tensor: torch.Tensor, bitwidth=4, codebook=None):
    """
    quantize tensor using k-means clustering
    :param fp32_tensor:
    :param bitwidth: [int] quantization bit width, default=4
    :param codebook: [Codebook] (the cluster centroids, the cluster label tensor)
    :return:
        [Codebook = (centroids, labels)]
            centroids: [torch.(cuda.)FloatTensor] the cluster centroids
            labels: [torch.(cuda.)LongTensor] cluster label tensor
    """
    if codebook is None:
        ############### YOUR CODE STARTS HERE ###############
        # bitwidth에 따라 클러스터 수를 설정하세요.
        n_clusters =
        ############### YOUR CODE ENDS HERE #################
        # k-means를 사용하여 quantization centroid를 얻습니다.
        kmeans = KMeans(n_clusters=n_clusters, mode='euclidean', verbose=0)
        labels = kmeans.fit_predict(fp32_tensor.view(-1, 1)).to(torch.long)
        centroids = kmeans.centroids.to(torch.float).view(-1)
        codebook = Codebook(centroids, labels)
    ############### YOUR CODE STARTS HERE ###############
    # 추론에서 사용할 quantized tensor를 구하기 위해 코드북을 디코딩하세요.
    quantized_tensor =
    ############### YOUR CODE ENDS HERE #################
    fp32_tensor.set_(quantized_tensor.view_as(fp32_tensor))
    return codebook

#### 채워야 할 답

1. `n_clusters = 2**bitwidth`
2. `quantized_tensor = codebook.centroids[codebook.labels]`

#### 간략 해설

1. k-means quantization에서 bitwidth가 표현 가능한 centroid 개수는 2^bitwidth입니다.
2. 각 원소를 자신이 속한 cluster의 centroid 값으로 치환해 양자화합니다.


### 문제 10. [실습 5] Quantization Aware Training 구현

- 원본 문제 파일/셀: `2. Quantization for CNN.ipynb` / code cell `96`
- 원본 답지 파일/셀: `2. Quantization for CNN_answer.ipynb` / code cell `96`


# [실습 5] Quantization Aware Training 구현

다음 코드북 업데이트 함수를 완성해 주세요.

**Hint**:

위의 centorid 업데이트 방정식은 동일한 클러스터 내에 있는 가중치들의 평균을 사용하여 centorid 값을 새로 업데이트하는 것입니다.

In [ ]:
def update_codebook(fp32_tensor: torch.Tensor, codebook: Codebook):
    """
    update the centroids in the codebook using updated fp32_tensor
    :param fp32_tensor: [torch.(cuda.)Tensor]
    :param codebook: [Codebook] (the cluster centroids, the cluster label tensor)
    """
    n_clusters = codebook.centroids.numel()
    fp32_tensor = fp32_tensor.view(-1)
    for k in range(n_clusters):
    ############### YOUR CODE STARTS HERE ###############
    # hint : torch.mean() 함수를 이용하여 평균을 구할 수 있습니다.
        codebook.centroids[k] =
    ############### YOUR CODE ENDS HERE #################

#### 채워야 할 답

1. `codebook.centroids[k] = torch.mean(fp32_tensor[codebook.labels==k])`

#### 간략 해설

1. QAT 중 같은 label에 속한 weight 평균으로 centroid를 갱신합니다.


## 3. Knowledge Distillation.ipynb

### 문제 11. [실습 2] Cosine Similarity 기반 KD 학습 함수 정의

- 원본 문제 파일/셀: `3. Knowledge Distillation.ipynb` / code cell `39`
- 원본 답지 파일/셀: `3. Knowledge Distillation_answer.ipynb` / code cell `39`


## [실습 2] Cosine Similarity 기반 KD 학습 함수 정의

이 함수는 **Teacher와 Student의 내부 표현(hidden representation)** 간 유사도를 **CosineEmbeddingLoss**를 통해 극대화하는 방식으로 Student를 학습시킵니다.  
이는 soft-label 기반 KD와 달리 **representation-level distillation**으로 분류되며, Student의 feature extractor 품질 향상에 초점을 둡니다.

In [ ]:
def train_cosine_loss(teacher,
                      student,
                      train_loader,
                      epochs,
                      learning_rate,
                      hidden_rep_loss_weight,
                      ce_loss_weight):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs * len(train_loader))

    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.cuda(), labels.cuda()

            optimizer.zero_grad()

            ##################### YOUR CODE STARTS HERE #####################
            # Forward pass with the teacher model and keep only the hidden representation
            with torch.no_grad():
                _, teacher_hidden_representation =

            # Forward pass with the student model
            student_logits, student_hidden_representation =

            # Calculate the cosine loss. Target is a vector of ones. From the loss formula above we can see that is
            # the case where loss minimization leads to cosine similarity increase.
            # Hint: cosine_loss(x, y, target)에서 target은 1로 이루어진 vector이며, torch.ones(inputs.size(0)).cuda())를 사용
            hidden_rep_loss =

            # Calculate the true label loss
            label_loss =

            # Weighted sum of the two losses
            loss =
            ##################### YOUR CODE ENDS HERE #######################

            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

#### 채워야 할 답

1. `_, teacher_hidden_representation = teacher(inputs)`
2. `student_logits, student_hidden_representation = student(inputs)`
3. `hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation,`
4. `target=torch.ones(inputs.size(0)).cuda())`
5. `label_loss = ce_loss(student_logits, labels)`
6. `loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss`

#### 간략 해설

1. teacher의 중간 representation을 student가 따라가도록 기준 feature를 얻습니다.
2. student 모델의 출력이 label loss와 distillation loss 계산에 사용됩니다.
3. teacher의 중간 representation을 student가 따라가도록 기준 feature를 얻습니다.
4. 문제 셀에서 비워진 핵심 구현 줄이며, 답지에서는 이 줄로 해당 알고리즘 단계를 완성합니다.
5. student 모델의 출력이 label loss와 distillation loss 계산에 사용됩니다.
6. 정답 label에 대한 일반 cross entropy loss를 함께 유지합니다.


### 문제 12. [실습 3] Hint-based Knowledge Distillation 학습 함수 정의 (MSE Loss 기반)

- 원본 문제 파일/셀: `3. Knowledge Distillation.ipynb` / code cell `52`
- 원본 답지 파일/셀: `3. Knowledge Distillation_answer.ipynb` / code cell `52`


## [실습 3] Hint-based Knowledge Distillation 학습 함수 정의 (MSE Loss 기반)

이 함수는 **중간 feature map**을 기준으로 Teacher와 Student의 표현을 정렬하기 위해 **Mean Squared Error (MSE) Loss**를 사용하는 Hint-based KD 학습 방식입니다.  
이를 통해 Student의 feature extractor가 Teacher의 중간 표현력을 모방하도록 유도합니다.


**학습 개념 요약**

| 손실 종류      | 역할                                       |
|----------------|--------------------------------------------|
| CrossEntropy   | 정답 라벨 기반 분류 학습                   |
| MSE Loss       | Teacher feature map ↔ Student regressed feature map 간 오차 최소화 |

In [ ]:
def train_mse_loss(teacher,
                   student,
                   train_loader,
                   epochs,
                   learning_rate,
                   feature_map_weight,
                   ce_loss_weight):
    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs * len(train_loader))

    teacher.eval()  # Teacher set to evaluation mode
    student.train() # Student to train mode

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.cuda(), labels.cuda()

            optimizer.zero_grad()

            ##################### YOUR CODE STARTS HERE #####################
            # Again ignore teacher logits
            with torch.no_grad():
                _, teacher_feature_map =

            # Forward pass with the student model
            student_logits, regressor_feature_map =

            # Calculate the loss
            hidden_rep_loss =

            # Calculate the true label loss
            label_loss =

            # Weighted sum of the two losses
            loss =
            ##################### YOUR CODE ENDS HERE #######################

            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

#### 채워야 할 답

1. `_, teacher_feature_map = teacher(inputs)`
2. `student_logits, regressor_feature_map = student(inputs)`
3. `hidden_rep_loss = mse_loss(regressor_feature_map, teacher_feature_map)`
4. `label_loss = ce_loss(student_logits, labels)`
5. `loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss`

#### 간략 해설

1. teacher의 중간 representation을 student가 따라가도록 기준 feature를 얻습니다.
2. student 모델의 출력이 label loss와 distillation loss 계산에 사용됩니다.
3. teacher의 중간 representation을 student가 따라가도록 기준 feature를 얻습니다.
4. student 모델의 출력이 label loss와 distillation loss 계산에 사용됩니다.
5. 정답 label에 대한 일반 cross entropy loss를 함께 유지합니다.


## 4. Pruning for LLM.ipynb

### 문제 13. [실습 1] Magnitude-based Pruning 구현

- 원본 문제 파일/셀: `4. Pruning for LLM.ipynb` / code cell `10`
- 원본 답지 파일/셀: `4. Pruning for LLM_answer.ipynb` / code cell `10`


## [실습 1] Magnitude-based Pruning 구현

Magnitude-based pruning 함수를 구현하세요. 구현은 CNN에서 magnitude-based pruning을 구현하는 방식과 유사합니다.

In [ ]:
@torch.no_grad()
def prune_magnitude(model, sparsity):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            ##################### YOUR CODE STARTS HERE #####################
            num_elements = 
            num_zeros = 
            importance = 
            threshold = 
            mask = 
            ##################### YOUR CODE ENDS HERE #######################
            W.mul_(mask)

#### 채워야 할 답

1. `num_elements = W.numel()`
2. `num_zeros = round(num_elements * sparsity)`
3. `importance = torch.abs(W)`
4. `threshold = torch.kthvalue(importance.flatten(), num_zeros)[0]`
5. `mask = importance > threshold`

#### 간략 해설

1. 문제 셀에서 비워진 핵심 구현 줄이며, 답지에서는 이 줄로 해당 알고리즘 단계를 완성합니다.
2. 목표 sparsity에 맞춰 제거할 weight 개수를 계산합니다.
3. Magnitude pruning은 절댓값이 작은 weight를 덜 중요하다고 보므로 절댓값을 importance로 사용합니다.
4. 목표 sparsity에 맞춰 제거할 weight 개수를 계산합니다.
5. threshold보다 큰 항목만 남기고 나머지는 pruning하기 위한 boolean mask입니다.


### 문제 14. [실습 2] Calibration 데이터셋 준비

- 원본 문제 파일/셀: `4. Pruning for LLM.ipynb` / code cell `16`
- 원본 답지 파일/셀: `4. Pruning for LLM_answer.ipynb` / code cell `16`


## [실습 2] Calibration 데이터셋 준비

[Wanda](https://arxiv.org/pdf/2306.11695)를 사용하여 importance를 계산하기 위해서는 calibration 데이터셋을 통해 activation을 추출해야 합니다.

이를 위해 ```get_calib_dataset``` 함수는 calibration dataset을 준비합니다.

In [ ]:
@torch.no_grad()
def get_calib_feat(model, tokenizer, samples):
    input_dict = dict()
    nsamples_dict = dict()
    def add_batch(m, x, y, name):
        if name not in input_dict:
            input_dict[name] = torch.zeros((m.weight.data.shape[1]), device=m.weight.data.device)
            nsamples_dict[name] = 0

        if isinstance(x, tuple):
            x = x[0]

        if len(x.shape) == 2:
            x = x.unsqueeze(0)
        tmp = x.shape[0]
        if len(x.shape) == 3:
            x = x.reshape((-1, x.shape[-1]))
        x = x.t()

        input_dict[name] *= nsamples_dict[name] / (nsamples_dict[name] + tmp)
        nsamples_dict[name] += tmp

        x = x.type(torch.float32)
        ##################### YOUR CODE STARTS HERE #####################
        # activation_norm을 계산하세요.
        # x.shape => (hidden_size, batch_size)
        activation_norm = 
        # activation_norm.shape => (hidden_size)
        ##################### YOUR CODE ENDS HERE #######################
        input_dict[name] += activation_norm / nsamples_dict[name]

    hooks = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in name:
            hooks.append(
                m.register_forward_hook(
                    partial(add_batch, name=name)))

    print("Collecting norm of input activations...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    pbar = tqdm.tqdm(samples)
    for input_ids in pbar:
        input_ids = input_ids.to(device)
        model(input_ids)

    for key in input_dict.keys():
        input_dict[key].sqrt_()

    for hook in hooks:
        hook.remove()
    return input_dict

#### 채워야 할 답

1. `activation_norm = torch.norm(x, p=2, dim=1) ** 2`

#### 간략 해설

1. 문제 셀에서 비워진 핵심 구현 줄이며, 답지에서는 이 줄로 해당 알고리즘 단계를 완성합니다.


### 문제 15. [실습 3] Wanda Pruning 구현

- 원본 문제 파일/셀: `4. Pruning for LLM.ipynb` / code cell `19`
- 원본 답지 파일/셀: `4. Pruning for LLM_answer.ipynb` / code cell `19`


## [실습 3] Wanda Pruning 구현

Wanda 논문을 참고하여 아래 빈 칸을 채워 Wanda Pruning을 구현하고 실행해보세요.

Wanda에서는 일반적인 magnitude-based pruning 방법과 다르게게 weight parameter의 행(row)별로 동일한 희소성(sparsity)으로 pruning을 진행합니다.

In [ ]:
@torch.no_grad()
def prune_wanda(model, sparsity, input_feat):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear) and "lm_head" not in n:
            W = m.weight.data
            ##################### YOUR CODE STARTS HERE #####################
            row, col = 
            num_zeros_per_row = 
            importance = 
            threshold = 
            mask = 
            ##################### YOUR CODE ENDS HERE #######################
            W.mul_(mask)

#### 채워야 할 답

1. `row, col = W.shape`
2. `num_zeros_per_row = round(col * sparsity)`
3. `importance = torch.abs(W) * input_feat[n]`
4. `threshold = torch.kthvalue(importance, num_zeros_per_row, dim=1)[0]`
5. `mask = importance > threshold.reshape(row, 1)`

#### 간략 해설

1. 문제 셀에서 비워진 핵심 구현 줄이며, 답지에서는 이 줄로 해당 알고리즘 단계를 완성합니다.
2. 목표 sparsity에 맞춰 제거할 weight 개수를 계산합니다.
3. Magnitude pruning은 절댓값이 작은 weight를 덜 중요하다고 보므로 절댓값을 importance로 사용합니다.
4. 목표 sparsity에 맞춰 제거할 weight 개수를 계산합니다.
5. threshold보다 큰 항목만 남기고 나머지는 pruning하기 위한 boolean mask입니다.


## 5. Quantization for LLM.ipynb

### 문제 16. [실습 1] Scale 1% salient channels

- 원본 문제 파일/셀: `5. Quantization for LLM.ipynb` / code cell `27`
- 원본 답지 파일/셀: `5. Quantization for LLM_answer.ipynb` / code cell `27`


# [실습 1] Scale 1% salient channels

1%의 가중치를 FP16으로 유지하면 모델 크기(총 비트 수로 측정)를 크게 늘리지 않고도 양자화 성능을 향상시킬 수 있지만, 이러한 혼합 정밀도 데이터 유형은 시스템 구현을 어렵게 만듭니다.

따라서 중요한 가중치를 실제로 FP16으로 유지하지 않고 중요한 가중치를 보호할 수 있는 방법을 찾아야 합니다.

In [ ]:
@torch.no_grad()
def pseudo_quantize_model_weight_scaleup(
    model, w_bit, q_group_size, input_feat, scale_factor
):
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            importance = sum(input_feat[n]).float()
            # 1퍼센트 채널의 개수
            num_samples = int(len(importance) * 0.01)

            ############### YOUR CODE STARTS HERE ###############

            # Step 1: importance를 기준으로 1%의 중요한 채널을 찾으세요  (hint: use torch.topk())
            # hint : torch.topk() 함수를 사용하세요. torch.topk() 함수는 PyTorch에서 텐서의 값 중 상위 k개의 값과 그들의 인덱스를 반환하는 함수입니다. torch.topk()[0]는 값을, torch.topk()[1]은 인덱스를 반환합니다.
            outlier_mask =

            ############### YOUR CODE ENDS HERE #################
            assert outlier_mask.dim() == 1

            ############### YOUR CODE STARTS HERE ###############

            # 스케일 팩터를 적용하는 것을 시뮬레이션하기 위해, 양자화 전에 스케일 팩터를 곱하고, 양자화 후에 스케일 팩터로 나눕니다.
            # scale_factor를 이용해 중요한 가중치 채널의 값을 확대합니다.
            m.weight.data[:, outlier_mask]

            m.weight.data = pseudo_quantize_tensor(m.weight.data, n_bit=w_bit, q_group_size=q_group_size)

            # Step 2: pseudo quantization이기 때문에 scale_factor를 이용해 중요한 가중치 채널의 값을 다시 축소하세요.
            m.weight.data[:, outlier_mask]

            ############### YOUR CODE ENDS HERE #################

#### 채워야 할 답

1. `outlier_mask = torch.topk(importance, int(len(importance) * 0.01))[1]`
2. `m.weight.data[:, outlier_mask] *= scale_factor`
3. `m.weight.data[:, outlier_mask] /= scale_factor`

#### 간략 해설

1. threshold보다 큰 항목만 남기고 나머지는 pruning하기 위한 boolean mask입니다.
2. 선택한 channel/weight를 scale up하여 quantization에서 중요한 값을 더 보존합니다.
3. scale 조정 후 수학적으로 같은 연산이 되도록 반대쪽 parameter를 보정합니다.


### 문제 17. [실습 2] Scale factor search

- 원본 문제 파일/셀: `5. Quantization for LLM.ipynb` / code cell `34`
- 원본 답지 파일/셀: `5. Quantization for LLM_answer.ipynb` / code cell `34`


# [실습 2] Scale factor search

지금까지 우리는 스케일링 팩터$s$를 직접 정의해 주었습니다.

그러나 Fine-tuning의 불안정성 때문에, 미리 정의된 검색 공간 내에서 최적의
$s$를 찾는 것이 더 나은 선택이 될 것입니다. 우리는 중요한 가중치를 보호하면서 다른 값을 고려하기 위해 검색 공간 내에서 최적의 스케일을 찾을 수 있습니다.

실제로, 논문에서는 활성화만 고려하는 것으로도 좋은 결과를 얻을 수 있음을 관찰할 수 있습니다.

우리는 스케일링 팩터 $s$를 활성화의 L1-norm (즉, acviation matrix의 절댓값들의 평균)의 $\alpha$제곱으로 설정할 것입니다.

$\alpha$의 값은 grid search를 통해 적절한 값으로 검색합니다.

검색을 위한 코드를 추가하고 실행하여 혼란도(perplexity)를 관찰하세요.

In [ ]:
@torch.no_grad()
def auto_scale_block(module, name, w_bit,
                     q_group_size,
                     input_feat):

    # find the best scale ratio
    def _search_module_scale(block, linears2scale: list, x, kwargs={}):

        x = x.to(next(block.parameters()).device)
        with torch.no_grad():
            org_out = block(x, **kwargs)
            if isinstance(org_out, tuple):
                org_out = org_out[0]

        s_x = x.view(-1, x.shape[-1]).abs().mean(0)
        s_x = torch.clamp(s_x, 1e-5)


        # Step 1: best_error, best_ratio, 및 best_scales를 초기화
        best_error = torch.inf
        best_ratio = -1
        best_scales = 0


        n_grid = 20
        history = []

        org_sd = {k: v.cpu() for k, v in block.state_dict().items()}
        for ratio in range(n_grid):
            # ratio is the \alpha in the formula
            ratio = ratio * 1 / n_grid

            ############### YOUR CODE STARTS HERE ###############

            # Step 2: 공식에 따라 스케일 계산
            scales =

            ############### YOUR CODE ENDS HERE #################
            assert scales.shape == s_x.shape

            scales = scales / (scales.max() * scales.min()).sqrt().view(1, -1)

            for fc in linears2scale:

                scales = scales.to(fc.weight.device)

                ############### YOUR CODE STARTS HERE ###############

                # Step 3: scale_factor를 이용해 중요한 가중치 채널의 값을 확대합니다.
                fc.weight.data

                fc.weight.data = pseudo_quantize_tensor(fc.weight.data, w_bit, q_group_size)

                # Step 4: scale_factor를 이용해 중요한 가중치 채널의 값을 다시 축소하세요.
                fc.weight.data

                ############### YOUR CODE ENDS HERE #################

            out = block(x, **kwargs)
            if isinstance(out, tuple):
                out = out[0]

            loss = (org_out - out).float().pow(2).mean().item()  # float prevents overflow
            history.append(loss)
            is_best = loss < best_error
            if is_best:
                best_error = loss
                best_ratio = ratio
                best_scales = scales
            block.load_state_dict(org_sd)

        if best_ratio == -1:
            print(history)
            raise Exception

        best_scales = best_scales.view(-1)

        assert torch.isnan(best_scales).sum() == 0, best_scales
        return best_scales.detach()

    # (1) Self-Attention 입력 부분 (Q, K, V projection)
    inp = input_feat[name + '.self_attn.out_proj']
    inp = torch.cat([x.unsqueeze(0) for x in inp], dim=0).unsqueeze(0)
    qkv = [module.self_attn.q_proj, module.self_attn.k_proj, module.self_attn.v_proj]
    final_scales = _search_module_scale(module.self_attn, qkv, inp)
    scale_ln_fcs(module.self_attn_layer_norm, qkv, final_scales)

    # (2) Self-Attention 출력 부분 (out_proj)
    inp = input_feat[name + '.self_attn.out_proj']
    inp = torch.cat([x.unsqueeze(0) for x in inp], dim=0)
    final_scales = _search_module_scale(module.self_attn.out_proj, [module.self_attn.out_proj], inp)
    scale_fc_fc(module.self_attn.v_proj, module.self_attn.out_proj, final_scales)

    # (3) Feed-Forward Network 첫 번째 FC (fc1)
    inp = input_feat[name + '.fc1']
    inp = torch.cat([x.unsqueeze(0) for x in inp], dim=0)
    final_scales = _search_module_scale(module.fc1, [module.fc1], inp)
    scale_ln_fcs(module.final_layer_norm, module.fc1, final_scales)

    # (4) Feed-Forward Network 두 번째 FC (fc2)
    inp = input_feat[name + '.fc2']
    inp = torch.cat([x.unsqueeze(0) for x in inp], dim=0)
    final_scales = _search_module_scale(module.fc2, [module.fc2], inp)
    scale_fc_fc(module.fc1, module.fc2, final_scales)

@torch.no_grad()
def pseudo_quantize_model_weight_auto_scale(model, w_bit, q_group_size, input_feat):
    """
    OPT 계열 모델 전체에 대해 자동 스케일 조정(auto-scale) 기반 pseudo quantization 수행.
    각 Decoder Layer 단위로 auto_scale_block을 적용.
    """
    from transformers.models.opt.modeling_opt import OPTDecoderLayer

    # 각 OPTDecoderLayer(Transformer 블록)에 대해 auto scaling 수행
    for name, module in model.named_modules():
        if isinstance(module, OPTDecoderLayer):
            auto_scale_block(module, name, w_bit, q_group_size, input_feat)

    # 모든 Linear layer에 pseudo quantization 최종 적용
    for n, m in model.named_modules():
        if isinstance(m, nn.Linear):
            m.weight.data = pseudo_quantize_tensor(
                m.weight.data, n_bit=w_bit, q_group_size=q_group_size
            )

#### 채워야 할 답

1. `scales = s_x ** ratio`
2. `fc.weight.data *= scales`
3. `fc.weight.data /= scales`

#### 간략 해설

1. activation scale에 ratio를 적용해 후보 scaling factor를 만듭니다.
2. 선택한 channel/weight를 scale up하여 quantization에서 중요한 값을 더 보존합니다.
3. scale 조정 후 수학적으로 같은 연산이 되도록 반대쪽 parameter를 보정합니다.


### 문제 18. [실습 3] Quantization difficulty migration

- 원본 문제 파일/셀: `5. Quantization for LLM.ipynb` / code cell `50`
- 원본 답지 파일/셀: `5. Quantization for LLM_answer.ipynb` / code cell `50`


# [실습 3] Quantization difficulty migration

실습을 통해 weight에는 s를 곱하고, activation에는 s를 나누어 quantization 난이도를 분배해 준 다음, quantization을 진행해 혼란도(perplexity)의 변화를 관찰해보세요.

일반적인 Transformer의 레이어 구조는 다음과 같습니다:

1. Self-Attention Block

    Input → LayerNorm → Query/Key/Value 생성 (FC) → Attention 연산 → Softmax → Attention Output

2. Feed-Forward Network (FFN)

    Attention Output → LayerNorm → FC1 → 활성화 함수 (ReLU, GELU 등) → FC2 → Output

그러므로 Transformer의 레이어에서 LayerNorm의 가중치에 스케일링 팩터 $s$를 나누고, FC의 가중치에 스케일링 팩터 $s$를 곱하면 weight에는 s를 곱하고, activation에는 s를 나누는 효과를 얻을 수 있습니다.

아래 코드를 수정하여 LayerNorm의 가중치에 스케일링 팩터 $s$를 나누고, FC의 가중치에 스케일링 팩터 $s$를 곱해주세요.

In [ ]:
@torch.no_grad()
def smooth_lm_by_scale(model, scale):
    """
    LLM(OPT 모델)의 각 Transformer Block에 대해 LayerNorm과 Linear layer 를 smoothing 하는 함수.
    - self-attention 블록의 (LN, q/k/v projection) 연결 구조와 feed-forward 블록의 (LN, fc1) 연결 구조에 동일한 스케일을 적용.
    """
    for name, module in model.named_modules():
        if isinstance(module, OPTDecoderLayer):
            # Self-Attention 부분 smoothing
            attn_ln = module.self_attn_layer_norm
            qkv = [
                module.self_attn.q_proj,
                module.self_attn.k_proj,
                module.self_attn.v_proj,
            ]
            smooth_ln_fcs_by_scale(attn_ln, qkv, scale)

            # Feed-Forward 부분 smoothing
            ffn_ln = module.final_layer_norm
            fc1 = module.fc1
            smooth_ln_fcs_by_scale(ffn_ln, fc1, scale)


@torch.no_grad()
def smooth_ln_fcs_by_scale(ln, fcs, scale):
    """
    LayerNorm(LN)과 그에 연결된 Fully Connected layer(FC)들 간의 파라미터 스케일을 맞추어 모델의 출력 분포를 안정화시키는 함수.

    내부 단계:
    1. LayerNorm의 weight, bias를 scale로 나눔 (출력 스케일 축소)
    2. 이후 연결된 FC layer의 weight를 동일 scale로 곱해 상쇄 (입력 스케일 복원)

    즉, forward 상에서 전체 스케일은 유지되면서 내부 파라미터 분포만 조정됨.
    """
    if not isinstance(fcs, list):
        fcs = [fcs]
    assert isinstance(ln, nn.LayerNorm)
    for fc in fcs:
        assert isinstance(fc, nn.Linear)

    ############### YOUR CODE STARTS HERE ###############
    # Step 1: layernorm의 weight와 bias를 scale로 나누어주세요.
    # div_() 함수는 텐서 자체를 스케일로 나누는 인플레이스 연산
    ln.weight
    ln.bias
    ############### YOUR CODE ENDS HERE #################

    for fc in fcs:
        ############### YOUR CODE STARTS HERE ###############
        # Step 2: fc의 weight에 scale을 곱해주세요.
        # mul_() 함수는 텐서 자체에 스케일을 곱하는 인플레이스 연산
        fc.weight
        ############### YOUR CODE ENDS HERE #################


#### 채워야 할 답

1. `ln.weight.div_(scale)`
2. `ln.bias.div_(scale)`
3. `fc.weight.mul_(scale)`

#### 간략 해설

1. scale 조정 후 수학적으로 같은 연산이 되도록 반대쪽 parameter를 보정합니다.
2. scale 조정 후 수학적으로 같은 연산이 되도록 반대쪽 parameter를 보정합니다.
3. 선택한 channel/weight를 scale up하여 quantization에서 중요한 값을 더 보존합니다.


### 문제 19. [실습 4] Scale factor sampling

- 원본 문제 파일/셀: `5. Quantization for LLM.ipynb` / code cell `55`
- 원본 답지 파일/셀: `5. Quantization for LLM_answer.ipynb` / code cell `55`


## [실습 4] Scale factor sampling

스케일링 팩터 $s$값의 설정에 따라 혼란도(perplexity)가 먼저 변화하는 것을 관찰했나요?


우리의 목표는 각 채널별 스케일링 팩터 s를 선택하여  X̂ = Xdiag(s)⁻¹가 양자화하기 쉽도록 만드는 것입니다.

양자화 오류를 줄이기 위해 모든 채널의 유효 양자화 비트를 늘려야 합니다.

가장 간단한 선택은 채널별로 서로 다른 스케일링 팩터를 설정하는 것입니다.

스케일링 팩터를 weight의 최대값으로 설정하면 weight의 양자화 난이도가 쉬워집니다. 그러나 activation의 양자화는 어려워집니다.

반대로, 스케일링 팩터를 activation의 최대값으로 설정하면 weight의 양자화가 어려워집니다.

우리는 weight와 activation의 양자화 난이도 사이에서 균형을 맞추기 위해 스케일링 팩터 s를 다음과 같이 설정합니다.

$s = \max(|X|)^{\alpha} / \max(|W|)^{1-\alpha}$

아래 코드를 수정하여 스케일링 팩터 s를 위의 식과 같이 설정해주세요.

In [ ]:
@torch.no_grad()
def smooth_ln_fcs(ln, fcs, act_scales, alpha=0.5):
    if not isinstance(fcs, list):
        fcs = [fcs]
    assert isinstance(ln, nn.LayerNorm)
    for fc in fcs:
        assert isinstance(fc, nn.Linear)
        assert ln.weight.numel() == fc.in_features == act_scales.numel()

    device, dtype = fcs[0].weight.device, fcs[0].weight.dtype
    act_scales = act_scales.to(device=device, dtype=dtype)
    weight_scales = torch.cat(
        [fc.weight.abs().max(dim=0, keepdim=True)[0] for fc in fcs], dim=0
    )
    weight_scales = weight_scales.max(dim=0)[0].clamp(min=1e-5)

    scales = (
        ############### YOUR CODE STARTS HERE ###############
        #Activation Scales 값과 Weight Scales 값에 alpha를 적절히 거듭제곱해주어야 합니다.

        ############### YOUR CODE ENDS HERE #################
    )

    scales.clamp(min=1e-5).to(device).to(dtype)

    ln.weight.div_(scales)
    ln.bias.div_(scales)

    for fc in fcs:
        fc.weight.mul_(scales.view(1, -1))


#### 채워야 할 답

1. `act_scales.pow(alpha) / weight_scales.pow(1 - alpha)`

#### 간략 해설

1. activation scale과 weight scale을 alpha로 절충해 SmoothQuant scaling factor를 계산합니다.


### 문제 20. [실습 5] Rotate Matrix 적용

- 원본 문제 파일/셀: `5. Quantization for LLM.ipynb` / code cell `67`
- 원본 답지 파일/셀: `5. Quantization for LLM_answer.ipynb` / code cell `67`


## [실습 5] Rotate Matrix 적용

R1, R2, R3, R4 등 다양한 Rotation Matrix가 Quantization 정확도를 높이기 위해 사용됩니다.

그러나, R2는 R1과 유사하게 적용 가능하고, R3와 R4는 Online에서 구해지는 Matrix이기 때문에 구현 난이도를 낮추기 위해 R1만 사용하는 것으로 하겠습니다.

QuaRot 혹은 SpinQuant 그림을 참고하셔서 각각의 연산에 R1이 어떻게 적용될 것인지 구현해보시기 바랍니다.

In [ ]:
def rotate_model_weight(model, R1):
    """
    모델의 모든 Embedding 및 Linear Layer 파라미터에 회전 행렬(R1)을 적용하여
    Weight Space Rotation을 수행하는 함수.

    - QuaRot, SpinQuant 등의 기법에서 사용되는 핵심 단계로, 모델의 weight를 직교 행렬로 회전시켜 Outlier 완화
    - 회전 행렬 R1은 (hidden_dim × hidden_dim) 형태의 직교 행렬이어야 함.
    """
    for n, m in model.named_modules():
        ############### YOUR CODE STARTS HERE ###############
        # Pytorch에서 '@' 연산자는 행렬 곱 (dot product)을 의미합니다.
        # nn.Linear의 weight 파라미터는 W^T 형태로 저장되어 있다는 점을 유의해야 합니다.
        #
        # Shape 요약:
        #   Embedding  : (num_tokens, hidden_dim)
        #   Linear     : (out_channels, in_channels)
        #   Rotation R : (hidden_dim, hidden_dim)

        if isinstance(m, nn.Embedding):
            # Embedding Weight: (num_tokens, hidden_dim)
            W_ = m.weight.data
            m.weight.data =

        if isinstance(m, nn.Linear):
            # Linear Layer: (out_features, in_features)
            if "o_proj" in n or "down_proj" in n:
                # Attention 출력 또는 FFN 다운프로젝션
                W_ = m.weight.data
                m.weight.data =
            else:
                # Q, K, V 프로젝션 및 FFN 게이트/업프로젝션
                W_ = m.weight.data
                m.weight.data =

        ############### YOUR CODE ENDS HERE #################

        # GPU 메모리 캐시 정리
        torch.cuda.empty_cache()

#### 채워야 할 답

1. `m.weight.data = W_ @ R1`
2. `m.weight.data = R1.T @ W_`
3. `m.weight.data = W_ @ R1`

#### 간략 해설

1. rotation matrix를 weight에 곱해 분포를 회전시키고 quantization 난이도를 낮춥니다.
2. rotation matrix를 weight에 곱해 분포를 회전시키고 quantization 난이도를 낮춥니다.
3. rotation matrix를 weight에 곱해 분포를 회전시키고 quantization 난이도를 낮춥니다.
